"Phase 3:Data Cleaning and Quality Assurance"  K Mohan Raju

In [1]:
import pandas as pd

listings = pd.read_csv('../data/raw/listings.csv.gz',
                       compression='gzip', low_memory=False)
reviews = pd.read_csv('../data/raw/reviews.csv.gz',
                      compression='gzip', parse_dates=['date'])

In [2]:
 keep_cols = [
    # identifiers and joins
    'id', 'host_id',
    # listing characteristics
    'property_type', 'room_type', 'accommodates', 'bedrooms', 'beds', 'bathrooms_text',
    # location
    'neighbourhood_cleansed', 'latitude', 'longitude',
    # pricing
    'price',
    # host attributes
    'hosts_time_as_host_years', 'host_is_superhost', 'host_listings_count',
    # availability and activity
    'minimum_nights', 'availability_365', 'number_of_reviews', 'reviews_per_month',
    'first_review', 'last_review', 'estimated_occupancy_l365d', 'estimated_revenue_l365d',
    # review scores
    'review_scores_rating', 'review_scores_cleanliness', 'review_scores_location', 'review_scores_value',
    # booking settings
    'instant_bookable']
print(len(keep_cols), "columns selected")
listings_selected = listings[keep_cols]

28 columns selected


In [3]:
[c for c in listings.columns if 'host' in c]

['host_id',
 'host_url',
 'host_profile_id',
 'host_profile_url',
 'host_name',
 'host_since',
 'hosts_time_as_user_years',
 'hosts_time_as_user_months',
 'hosts_time_as_host_years',
 'hosts_time_as_host_months',
 'host_location',
 'host_about',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'host_thumbnail_url',
 'host_picture_url',
 'host_neighbourhood',
 'host_listings_count',
 'host_total_listings_count',
 'host_verifications',
 'host_has_profile_pic',
 'host_identity_verified',
 'calculated_host_listings_count',
 'calculated_host_listings_count_entire_homes',
 'calculated_host_listings_count_private_rooms',
 'calculated_host_listings_count_shared_rooms']

In [4]:
print("Duplicate listing ids:", listings_selected['id'].duplicated().sum())
print("Duplicate full rows:", listings_selected.duplicated().sum())

Duplicate listing ids: 0
Duplicate full rows: 0


In [5]:
audit = pd.DataFrame({
    'dtype': listings_selected.dtypes.astype(str),
    'pct_missing': (listings_selected.isna().mean() *100).round(1),
    'n_unique': listings_selected.nunique(),
    'example': listings_selected.iloc[0]
}).sort_values('pct_missing',ascending=False)
audit

,dtype,pct_missing,n_unique,example
instant_bookable,float64,100.0,0,NaN
beds,float64,35.1,28,3.0
estimated_revenue_l365d,float64,32.8,16782,12898.0
price,object,32.8,11970,$234.50
bedrooms,float64,25.9,21,1.0
review_scores_cleanliness,float64,23.7,201,4.57
review_scores_location,float64,23.7,159,4.54
last_review,object,23.7,3792,2026-06-05
first_review,object,23.7,4832,2010-03-21
reviews_per_month,float64,23.7,895,0.99


###  Quality audit findings 
- No duplicate listings ids or rows (0 / 0) - id is safe as the primary key
- instant _bookable: 100% empty in this snapshot - removed from selection
- price: stored as text with "$"and ","; 32.8%missing. estimated_revenue_1365d missing the identical 32.8% because it derives from price
- All review-related columns missingtogether at 23.7% - the signature of never-reviewed listings; missing is informative, not an error
- first_review / last_review stored as text - need date conversion
- beds 35.1% and bedrooms 25.9% missing - treatme t decided below
- host_is_superhost coded "t"/"f" -needs boolean conversion
- property_type has 91 categories - needs grouping


##  Cleaning decision  1 : price
** What I found: **

** What I decided: **

** Why: **


In [6]:
listings_clean = listings_selected.copy()
listings_clean['price'] = (listings_clean['price']
                           .str.replace('$', '', regex=False)
                           .str.replace(',', '', regex=False)
                           .astype(float))
listings_clean['price'].describe()

count     62240.000000
mean        271.469220
std        2190.599006
min           2.230000
25%         100.000000
50%         180.000000
75%         300.032500
max      527524.000000
Name: price, dtype: float64

In [7]:
print("Listings over £1,000/night:", (listings_clean['price'] > 1000).sum())
print("Listings over £10,000/night:", (listings_clean['price'] > 10000).sum())
listings_clean.loc[listings_clean['price'] > 10000,
                   ['id', 'property_type', 'room_type', 'accommodates',
                    'price', 'number_of_reviews']].sort_values('price', ascending=False).head(10)

Listings over £1,000/night: 1277
Listings over £10,000/night: 21


,id,property_type,room_type,accommodates,price,number_of_reviews
67130,1389784758661918909,Entire rental unit,Entire home/apt,1,527524.0,0
8400,13254774,Private room in rental unit,Private room,1,75000.0,97
61588,1289811437722717777,Private room in rental unit,Private room,3,59214.0,2
57238,1203041287118705621,Private room in rental unit,Private room,3,28000.0,11
56041,1183168933886992107,Private room in rental unit,Private room,3,23686.0,4
57946,1207889725595849665,Entire villa,Entire home/apt,16,19150.0,0
64156,1342870516779702858,Entire rental unit,Entire home/apt,2,16316.8,1
56739,1196233560198207804,Entire home,Entire home/apt,16,14744.0,0
66648,1384148619625788404,Entire rental unit,Entire home/apt,2,13600.0,0
34627,644043613421252064,Entire serviced apartment,Entire home/apt,10,13209.0,0


**What I found:** price was stored as text... and 32.8% missing... max of £527,524 indicates data-entry outliers

**What I decided:** ...

**Why:** ...

### Cleaning: host_is_superhost

**What I found:** coded as text "t"/"f", with 76 rows (0.1%) missing.

**What I decided:** mapped to True/False. The 76 missing stay as NaN.

**Why:** boolean type enables direct filtering and group comparison. Unknown status is not the same as "not a superhost", so imputing False would invent data. Result: 17,559 superhosts (19%) vs 75,003 — this imbalance is noted for the ML phase.

In [8]:
listings_clean['host_is_superhost'] = listings_clean['host_is_superhost'].map({'t': True, 'f': False})
listings_clean['host_is_superhost'].value_counts(dropna=False)

host_is_superhost
False    75003
True     17559
NaN         76
Name: count, dtype: int64

### Cleaning: first_review / last_review

**What I found:** stored as text (object dtype).

**What I decided:** converted with pd.to_datetime. Never-reviewed listings (23.7%) become NaT.

**Why:** date arithmetic is needed for the is_active feature and trend analysis. NaT correctly represents "no review has ever happened".

In [9]:
listings_clean['first_review'] = pd.to_datetime(listings_clean['first_review'])
listings_clean['last_review'] = pd.to_datetime(listings_clean['last_review'])
listings_clean[['first_review', 'last_review']].dtypes

first_review    datetime64[ns]
last_review     datetime64[ns]
dtype: object

In [10]:
listings_clean.dtypes

id                                    int64
host_id                               int64
property_type                        object
room_type                            object
accommodates                          int64
bedrooms                            float64
beds                                float64
bathrooms_text                       object
neighbourhood_cleansed               object
latitude                            float64
longitude                           float64
price                               float64
hosts_time_as_host_years            float64
host_is_superhost                    object
host_listings_count                 float64
minimum_nights                      float64
availability_365                      int64
number_of_reviews                     int64
reviews_per_month                   float64
first_review                 datetime64[ns]
last_review                  datetime64[ns]
estimated_occupancy_l365d             int64
estimated_revenue_l365d         

### Feature 1: bathrooms (extracted from bathrooms_text)

**What I found:** bathrooms_text holds values like "1 bath", "1.5 baths", "Half-bath" — text, unusable for numeric analysis.

**Business rationale:** bathroom count is a core property attribute that plausibly influences price and guest ratings; extracting it as a number makes it usable for correlation, ML features, and Power BI comparisons.

In [11]:
listings_clean['bathrooms'] = (listings_clean['bathrooms_text']
                               .str.extract(r'(\d+\.?\d*)')
                               .astype(float))
listings_clean[['bathrooms_text', 'bathrooms']].drop_duplicates().head(15)

,bathrooms_text,bathrooms
0,1 bath,1.0
1,1 shared bath,1.0
3,2 baths,2.0
4,1.5 baths,1.5
5,NaN,NaN
9,3 baths,3.0
10,1 private bath,1.0
24,1.5 shared baths,1.5
28,2 shared baths,2.0
31,0 shared baths,0.0


In [12]:
listings_clean.loc[listings_clean['bathrooms_text'].str.contains('alf', na=False),
                   ['bathrooms_text', 'bathrooms']].value_counts()

Series([], Name: count, dtype: int64)

In [13]:
listings_clean.loc[listings_clean['bathrooms_text'].str.contains('alf', na=False), 'bathrooms'] = 0.5

In [14]:
listings_clean['bathrooms'].describe()

count    92504.000000
mean         1.331705
std          0.688909
min          0.000000
25%          1.000000
50%          1.000000
75%          1.500000
max         30.000000
Name: bathrooms, dtype: float64

In [15]:
listings_clean['price_per_person'] = listings_clean['price'] / listings_clean['accommodates']
listings_clean['price_per_person'].describe()

count     62240.000000
mean         87.169404
std        2141.970632
min           1.066667
25%          41.000000
50%          58.500000
75%          86.000000
max      527524.000000
Name: price_per_person, dtype: float64

In [16]:
snapshot_date = pd.Timestamp('2026-06-30')
listings_clean['is_active'] = listings_clean['last_review'] >= snapshot_date - pd.DateOffset(months=12)
listings_clean['is_active'].value_counts(dropna=False)

is_active
True     46880
False    45758
Name: count, dtype: int64

In [17]:
top_types = listings_clean['property_type'].value_counts().head(10).index
listings_clean['property_type_grouped'] = listings_clean['property_type'].where(
    listings_clean['property_type'].isin(top_types), 'Other')
listings_clean['property_type_grouped'].value_counts()

property_type_grouped
Entire rental unit             40129
Private room in rental unit    12907
Private room in home           10846
Entire home                     9153
Entire condo                    7935
Other                           3400
Private room in condo           3047
Entire serviced apartment       1862
Room in hotel                   1253
Private room in townhouse       1097
Entire townhouse                1009
Name: count, dtype: int64

### Price Policy: 99th Percentile Cap

Based on the charter commitment, extreme prices above the 99th percentile are capped to prevent the tail from distorting means and ML models. The unpriced listings (32.8%) are retained as NaN because missing prices usually reflect no upcoming availability, not data error, and these listings remain valid for occupancy and review analysis.

In [18]:
price_99th = listings_clean['price'].quantile(0.99)
print(f"99th percentile price: £{price_99th:.2f}")
listings_clean['price'] = listings_clean['price'].clip(upper=price_99th)
listings_clean['price'].describe()

99th percentile price: £1373.46


count    62240.000000
mean       244.360527
std        225.066039
min          2.230000
25%        100.000000
50%        180.000000
75%        300.032500
max       1373.457500
Name: price, dtype: float64

In [19]:
missing_by_borough = listings_clean.groupby('neighbourhood_cleansed')['price'].apply(lambda s: (s.isna().sum() / len(s) * 100)).sort_values(ascending=False)
print("Boroughs with highest price missingness:")
print(missing_by_borough.head(10))

Boroughs with highest price missingness:
neighbourhood_cleansed
Hackney                 49.235580
Islington               43.073684
Lambeth                 40.730337
Tower Hamlets           40.529085
Lewisham                39.452372
Haringey                38.736591
Southwark               38.653995
Wandsworth              38.055909
Waltham Forest          36.353830
Richmond upon Thames    35.003978
Name: price, dtype: float64


In [20]:
reviews_monthly = (reviews
                   .assign(year_month=reviews['date'].dt.to_period('M').astype(str))
                   .groupby(['listing_id', 'year_month'])
                   .size()
                   .reset_index(name='review_count'))
print("Shape:", reviews_monthly.shape)
reviews_monthly.to_csv('../data/reviews_monthly.csv', index=False)

Shape: (900540, 3)


In [21]:
reviews_clean = reviews[reviews['date'] >= '2024-07-01'].copy()
reviews_clean = reviews_clean.dropna(subset=['comments'])
reviews_clean = reviews_clean[['listing_id', 'id', 'date', 'reviewer_id', 'comments']]

print("Shape:", reviews_clean.shape)
reviews_clean.to_csv('../data/reviews_clean.csv', index=False)

Shape: (978778, 5)


In [22]:
drop_for_export = ['bathrooms_text']  # replaced by bathrooms
listings_clean.drop(columns=drop_for_export).to_csv('../data/listings_clean.csv', index=False)
print("Exported:", listings_clean.shape)
print("Max price in export:", listings_clean['price'].max())

Exported: (92638, 32)
Max price in export: 1373.4575000000004
